# CNN

In [2]:
# -*- coding: utf-8 -*-
# WISDM (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Split: random window-level split (train/val/test)
# - Train-only normalization: fit scaler on TRAIN windows only (raw space), then apply to all
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)

import os, random, copy
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    wisdm_txt_path: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/WISDM_ar_v1.1_raw.txt"  # <-- change
    window_size: int = 80
    step_size: int = 40

    # split
    test_ratio: float = 0.2
    val_ratio_within_train: float = 0.2  # val split from train portion

    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4

    d_model: int = 128

    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) WISDM Loader (single txt) - stores RAW windows too
# ==============================================================================
class WISDMDataset(Dataset):
    """
    Creates windows from WISDM raw txt.
    Stores:
      - self.X_raw: (N, C, T) raw float32
      - self.X    : (N, C, T) working array (raw initially, becomes normalized after apply_scaler)
      - self.y    : (N,)
      - self.subjects : (N,)
    """
    def __init__(self, file_path: str, window_size: int = 80, step_size: int = 40):
        super().__init__()
        self.file_path = file_path
        self.window_size = window_size
        self.step_size = step_size

        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"WISDM txt file not found: {file_path}")

        df = self._load_file(file_path)
        X, y, s = self._create_windows(df)

        self.X_raw = X.astype(np.float32)           # (N,C,T)
        self.X = self.X_raw.copy().astype(np.float32)
        self.y = y.astype(np.int64)
        self.subjects = s.astype(np.int64)
        self.unique_subjects = sorted(np.unique(self.subjects))
        self.n_classes = int(len(np.unique(self.y)))

        print("=" * 80)
        print("Loaded WISDM dataset (single txt)")
        print(f"  X_raw shape   : {self.X_raw.shape}  (N, C, T)")
        print(f"  y shape       : {self.y.shape}  (N,)")
        print(f"  subjects shape: {self.subjects.shape} (N,)")
        print(f"  num classes   : {self.n_classes}")
        print(f"  unique subjects: {self.unique_subjects[:10]} ... (total {len(self.unique_subjects)})")
        print("=" * 80)

    def _load_file(self, file_path: str) -> pd.DataFrame:
        WISDM_LABEL_MAP = {
            "walking": 0,
            "jogging": 1,
            "sitting": 2,
            "standing": 3,
            "upstairs": 4,
            "downstairs": 5,
        }

        with open(file_path, "r") as f:
            lines = f.readlines()

        rows = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            line = line.replace(";", "")
            parts = line.split(",")

            if len(parts) != 6:
                continue

            subj, act, ts, x, y, z = parts
            if x.strip() == "" or y.strip() == "" or z.strip() == "":
                continue

            act_norm = act.strip().lower()
            if act_norm not in WISDM_LABEL_MAP:
                continue

            rows.append([subj, act_norm, ts, x, y, z])

        if not rows:
            raise ValueError(f"No valid rows parsed from file: {file_path}")

        df = pd.DataFrame(rows, columns=["subject", "activity", "timestamp", "x", "y", "z"])
        df = df.replace(["", "NaN", "nan"], np.nan).dropna(subset=["subject", "x", "y", "z"])

        df["subject"] = pd.to_numeric(df["subject"], errors="coerce")
        df["x"] = pd.to_numeric(df["x"], errors="coerce")
        df["y"] = pd.to_numeric(df["y"], errors="coerce")
        df["z"] = pd.to_numeric(df["z"], errors="coerce")
        df = df.dropna(subset=["subject", "x", "y", "z"])
        if df.empty:
            raise ValueError("After cleaning, WISDM DataFrame is empty. Check file format.")

        df["subject"] = df["subject"].astype(int)
        df["activity_id"] = df["activity"].map(WISDM_LABEL_MAP).astype(int)
        return df

    def _create_windows(self, df: pd.DataFrame):
        X_list, y_list, s_list = [], [], []

        for subj_id in sorted(df["subject"].unique()):
            df_sub = df[df["subject"] == subj_id]
            data = df_sub[["x", "y", "z"]].to_numpy(dtype=np.float32)     # (L,3)
            labels = df_sub["activity_id"].to_numpy(dtype=np.int64)       # (L,)
            L = len(df_sub)

            start = 0
            while start + self.window_size <= L:
                end = start + self.window_size
                window_x = data[start:end]           # (T,3)
                window_y = labels[end - 1]           # label at end

                X_list.append(window_x.T)            # (3,T)
                y_list.append(window_y)
                s_list.append(subj_id)

                start += self.step_size

        if len(X_list) == 0:
            raise ValueError("[WISDMDataset] No windows created. Try smaller window_size or check data.")

        X = np.stack(X_list, axis=0).astype(np.float32)  # (N,3,T)
        y = np.array(y_list, dtype=np.int64)
        s = np.array(s_list, dtype=np.int64)
        return X, y, s

    def fit_scaler(self, indices: np.ndarray) -> StandardScaler:
        """Fit StandardScaler on RAW train windows only."""
        Xtr = self.X_raw[indices]  # (N,C,T)
        N, C, T = Xtr.shape
        X2 = np.transpose(Xtr, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
        scaler = StandardScaler()
        scaler.fit(X2)
        self.scaler = scaler
        return scaler

    def apply_scaler(self, scaler: StandardScaler = None):
        """Apply scaler to RAW windows and write into self.X as normalized windows (N,C,T)."""
        if scaler is None:
            scaler = getattr(self, "scaler", None)
        assert scaler is not None, "Scaler is not fitted. Call fit_scaler() first."

        X = self.X_raw
        N, C, T = X.shape
        X2 = np.transpose(X, (0, 2, 1)).reshape(-1, C)
        X2 = scaler.transform(X2)
        self.X = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)  # (N,C,T)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx: int):
        # return normalized X by default (self.X), shape (C,T)
        return (
            torch.from_numpy(self.X[idx]).float(),   # (C,T)
            torch.tensor(self.y[idx]).long(),
            int(self.subjects[idx]),
        )


# ==============================================================================
# 4) Helpers: subset dataset -> model input (B,T,C)
# ==============================================================================
class SubsetForModel(Dataset):
    """
    Wrap a base WISDMDataset + indices.
    Converts X from (C,T) to (T,C) to match your model forward(x_btc).
    Returns (T,C), y
    """
    def __init__(self, base: WISDMDataset, indices: np.ndarray):
        self.base = base
        self.indices = np.array(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        x_ct, y, _s = self.base[int(self.indices[i])]
        x_tc = x_ct.transpose(0, 1).contiguous()  # (T,C)
        return x_tc, y


# ==============================================================================
# 5) Models (same as your UCI code, but in_ch=3, num_classes from dataset)
# ==============================================================================
class ConvBNAct(nn.Module):
    def __init__(self, c_in, c_out, k, s=1, p=None, g=1):
        super().__init__()
        self.c = nn.Conv1d(c_in, c_out, k, s, k//2 if p is None else p, groups=g, bias=False)
        self.bn = nn.BatchNorm1d(c_out)
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(self.bn(self.c(x)))

class MultiPathCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=128, branches=(3,5,9,15), stride=2):
        super().__init__()
        h = d_model // 2
        self.pre = ConvBNAct(in_ch, h, 1)
        self.branches = nn.ModuleList([
            nn.Sequential(ConvBNAct(h, h, k, stride, g=h), ConvBNAct(h, h, 1))
            for k in branches
        ])
        self.post = ConvBNAct(len(branches)*h, d_model, 1)

    def forward(self, x_bct):
        z = self.pre(x_bct)
        zs = [b(z) for b in self.branches]
        return self.post(torch.cat(zs, dim=1))  # (B, D, T')

class GAPModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=3):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.fc = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)     # (B,C,T)
        fmap = self.backbone(x)       # (B,D,T')
        feat = fmap.transpose(1, 2)   # (B,T',D)
        pooled = feat.mean(dim=1)     # (B,D)
        return self.fc(pooled)

class ProductionTPA(nn.Module):
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1, temperature=0.07):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)
        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_btd):
        B, T, D = x_btd.shape
        P = self.num_prototypes

        x = self.pre_norm(x_btd)
        K = self.k_proj(x)
        V = self.v_proj(x)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, L):
            return t.view(B, L, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)   # (B,H,P,dh)
        Kh = split_heads(K, T)    # (B,H,T,dh)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature  # (B,H,P,T)
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)  # (B,H,P,dh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)
        z_tpa = proto_tokens.mean(dim=1)       # (B,D)
        return self.fuse(z_tpa)

class TPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=3, tpa_config=None):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config["num_prototypes"],
            heads=tpa_config["heads"],
            dropout=tpa_config["dropout"],
            temperature=tpa_config["temperature"],
        )
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)     # (B,C,T)
        fmap = self.backbone(x)       # (B,D,T')
        feat = fmap.transpose(1, 2)   # (B,T',D)
        z = self.tpa(feat)            # (B,D)
        return self.classifier(z)

class GatedTPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=6, in_ch=3, tpa_config=None):
        super().__init__()
        self.backbone = MultiPathCNN(in_ch=in_ch, d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config["num_prototypes"],
            heads=tpa_config["heads"],
            dropout=tpa_config["dropout"],
            temperature=tpa_config["temperature"],
        )
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, x_btc):
        x = x_btc.transpose(1, 2)     # (B,C,T)
        fmap = self.backbone(x)       # (B,D,T')
        feat = fmap.transpose(1, 2)   # (B,T',D)

        z_gap = feat.mean(dim=1)
        z_tpa = self.tpa(feat)

        g = self.gate(torch.cat([z_gap, z_tpa], dim=-1))
        z = g * z_gap + (1.0 - g) * z_tpa
        return self.classifier(z)

def create_model(model_name: str, cfg: Config, num_classes: int, in_ch: int = 3):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )
    if model_name == "GAP":
        return GAPModel(d_model=cfg.d_model, num_classes=num_classes, in_ch=in_ch).to(cfg.device)
    if model_name == "TPA":
        return TPAModel(d_model=cfg.d_model, num_classes=num_classes, in_ch=in_ch, tpa_config=tpa_config).to(cfg.device)
    if model_name == "Gated-TPA":
        return GatedTPAModel(d_model=cfg.d_model, num_classes=num_classes, in_ch=in_ch, tpa_config=tpa_config).to(cfg.device)
    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()  # (B,T,C)
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space, X is (N,C,T))
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    # channel-wise sigma on this batch
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn

def apply_scaler_to_array(X_raw_nct: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    """X_raw_nct: (N,C,T) -> normalized (N,C,T) using fitted scaler"""
    N, C, T = X_raw_nct.shape
    X2 = np.transpose(X_raw_nct, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
    X2 = scaler.transform(X2)
    Xn = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model (based on its clean F1)
# ==============================================================================
def run_wisdm_experiment(cfg: Config):
    # ---- Load full dataset (raw windows created inside) ----
    full_dataset = WISDMDataset(cfg.wisdm_txt_path, window_size=cfg.window_size, step_size=cfg.step_size)
    num_classes = full_dataset.n_classes
    assert full_dataset.X_raw.shape[1] == 3, "WISDM expected 3 channels (x,y,z)."

    n_total = len(full_dataset)
    indices_all = np.arange(n_total)

    # ---- Window-level random split: train vs test (stratified) ----
    tr_idx, te_idx = train_test_split(
        indices_all,
        test_size=cfg.test_ratio,
        random_state=SEED,
        stratify=full_dataset.y
    )

    # ---- Split train into train/val (stratified) ----
    tr2_idx, va_idx = train_test_split(
        tr_idx,
        test_size=cfg.val_ratio_within_train,
        random_state=SEED,
        stratify=full_dataset.y[tr_idx]
    )

    # ---- Train-only scaler on RAW train windows ----
    scaler = full_dataset.fit_scaler(tr2_idx)

    # ---- Apply scaler to entire dataset (writes normalized to full_dataset.X) ----
    full_dataset.apply_scaler(scaler)

    # ---- Build loaders (return (T,C), y) ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_ds = SubsetForModel(full_dataset, tr2_idx)
    val_ds   = SubsetForModel(full_dataset, va_idx)
    test_ds  = SubsetForModel(full_dataset, te_idx)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (same list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # ---- Raw test windows (N,C,T) from stored X_raw ----
    X_test_raw_nct = full_dataset.X_raw[te_idx].astype(np.float32)  # raw
    y_test = full_dataset.y[te_idx].astype(np.int64)
    rng = np.random.default_rng(SEED)

    @torch.no_grad()
    def eval_model_on_raw_perturbed(model, Xpert_raw_nct: np.ndarray) -> Tuple[float, float]:
        # normalize using train scaler, then make loader
        Xnorm_nct = apply_scaler_to_array(Xpert_raw_nct, scaler)       # (N,C,T)
        # convert to (N,T,C) for model
        Xnorm_ntc = np.transpose(Xnorm_nct, (0, 2, 1)).astype(np.float32)

        ds = torch.utils.data.TensorDataset(
            torch.from_numpy(Xnorm_ntc).float(),
            torch.from_numpy(y_test).long()
        )
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("WISDM | CLEAN + ROBUSTNESS (PER MODEL)")
    print(f"Splits: total={n_total} | train={len(tr2_idx)} | val={len(va_idx)} | test={len(te_idx)}")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg, num_classes=num_classes, in_ch=3)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_raw_nct, sp)
            acc, f1 = eval_model_on_raw_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_raw_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_raw_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_raw_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_wisdm_experiment(cfg)

Loaded WISDM dataset (single txt)
  X_raw shape   : (27108, 3, 80)  (N, C, T)
  y shape       : (27108,)  (N,)
  subjects shape: (27108,) (N,)
  num classes   : 6
  unique subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ... (total 36)

WISDM | CLEAN + ROBUSTNESS (PER MODEL)
Splits: total=27108 | train=17348 | val=4338 | test=5422

--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.9864 | F1=0.9794 | (Val best F1=0.9772)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9710 | F1=0.9625 | ΔF1=+0.0169
speed=0.90 | Acc=0.9672 | F1=0.9577 | ΔF1=+0.0217
speed=0.95 | Acc=0.9600 | F1=0.9503 | ΔF1=+0.0291
speed=1.05 | Acc=0.9522 | F1=0.9418 | ΔF1=+0.0376
speed=1.10 | Acc=0.9517 | F1=0.9412 | ΔF1=+0.0382
speed=1.15 | Acc=0.9463 | F1=0.9359 | ΔF1=+0.0436

[2) Additive Gaussian N

# BiLSTM

In [3]:
# -*- coding: utf-8 -*-
# WISDM (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Train/Val/Test: random window-level split (train-only normalization: fit on TRAIN windows only)
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)
#
# IMPORTANT:
#   - Model code (BiLSTMBackbone/GAP/TPA/Gated-TPA/ProductionTPA) is kept as-is in structure.
#   - Only minimal wiring changes for WISDM: in_ch=3, num_classes=6, and dataset/scaler/split.
#   - Input to model is (B, T, C). WISDM windows are stored as (N, C, T) internally and converted to (T, C).

import os, random, copy
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    # ---- WISDM ----
    wisdm_txt_path: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/WISDM_ar_v1.1_raw.txt"  # <-- CHANGE THIS
    window_size: int = 80
    step_size: int = 40

    test_ratio: float = 0.2
    val_ratio_within_train: float = 0.2  # val split from (train portion)

    # ---- Train ----
    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4

    d_model: int = 128

    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) WISDM Dataset (single txt) + train-only scaler
# ==============================================================================
class WISDMDataset(Dataset):
    """
    Creates windows from WISDM raw txt.

    Stores:
      - self.X_raw: (N, C, T) raw float32
      - self.X    : (N, C, T) working array (raw initially, becomes normalized after apply_scaler)
      - self.y    : (N,)
      - self.subjects : (N,)
    """
    def __init__(self, file_path: str, window_size: int = 80, step_size: int = 40):
        super().__init__()
        self.file_path = file_path
        self.window_size = window_size
        self.step_size = step_size

        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"WISDM txt file not found: {file_path}")

        df = self._load_file(file_path)
        X, y, s = self._create_windows(df)

        self.X_raw = X.astype(np.float32)                 # (N,C,T)
        self.X = self.X_raw.copy().astype(np.float32)     # working tensor (will be normalized)
        self.y = y.astype(np.int64)
        self.subjects = s.astype(np.int64)
        self.unique_subjects = sorted(np.unique(self.subjects))
        self.n_classes = int(len(np.unique(self.y)))

        print("=" * 80)
        print("Loaded WISDM dataset (single txt)")
        print(f"  X_raw shape   : {self.X_raw.shape}  (N, C, T)")
        print(f"  y shape       : {self.y.shape}  (N,)")
        print(f"  subjects shape: {self.subjects.shape} (N,)")
        print(f"  num classes   : {self.n_classes}")
        print(f"  unique subjects: {self.unique_subjects[:10]} ... (total {len(self.unique_subjects)})")
        print("=" * 80)

    def _load_file(self, file_path: str) -> pd.DataFrame:
        WISDM_LABEL_MAP = {
            "walking": 0,
            "jogging": 1,
            "sitting": 2,
            "standing": 3,
            "upstairs": 4,
            "downstairs": 5,
        }

        with open(file_path, "r") as f:
            lines = f.readlines()

        rows = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            line = line.replace(";", "")
            parts = line.split(",")

            if len(parts) != 6:
                continue

            subj, act, ts, x, y, z = parts
            if x.strip() == "" or y.strip() == "" or z.strip() == "":
                continue

            act_norm = act.strip().lower()
            if act_norm not in WISDM_LABEL_MAP:
                continue

            rows.append([subj, act_norm, ts, x, y, z])

        if not rows:
            raise ValueError(f"No valid rows parsed from file: {file_path}")

        df = pd.DataFrame(rows, columns=["subject", "activity", "timestamp", "x", "y", "z"])
        df = df.replace(["", "NaN", "nan"], np.nan).dropna(subset=["subject", "x", "y", "z"])

        df["subject"] = pd.to_numeric(df["subject"], errors="coerce")
        df["x"] = pd.to_numeric(df["x"], errors="coerce")
        df["y"] = pd.to_numeric(df["y"], errors="coerce")
        df["z"] = pd.to_numeric(df["z"], errors="coerce")
        df = df.dropna(subset=["subject", "x", "y", "z"])
        if df.empty:
            raise ValueError("After cleaning, WISDM DataFrame is empty. Check file format.")

        df["subject"] = df["subject"].astype(int)
        df["activity_id"] = df["activity"].map(WISDM_LABEL_MAP).astype(int)
        return df

    def _create_windows(self, df: pd.DataFrame):
        X_list, y_list, s_list = [], [], []

        for subj_id in sorted(df["subject"].unique()):
            df_sub = df[df["subject"] == subj_id]
            data = df_sub[["x", "y", "z"]].to_numpy(dtype=np.float32)  # (L,3)
            labels = df_sub["activity_id"].to_numpy(dtype=np.int64)    # (L,)
            L = len(df_sub)

            start = 0
            while start + self.window_size <= L:
                end = start + self.window_size
                window_x = data[start:end]     # (T,3)
                window_y = labels[end - 1]     # label at end

                X_list.append(window_x.T)      # (3,T)
                y_list.append(window_y)
                s_list.append(subj_id)
                start += self.step_size

        if len(X_list) == 0:
            raise ValueError("[WISDMDataset] No windows created. Try smaller window_size or check data.")

        X = np.stack(X_list, axis=0).astype(np.float32)  # (N,3,T)
        y = np.array(y_list, dtype=np.int64)
        s = np.array(s_list, dtype=np.int64)
        return X, y, s

    def fit_scaler(self, indices: np.ndarray) -> StandardScaler:
        """Fit StandardScaler on RAW train windows only."""
        Xtr = self.X_raw[indices]  # (N,C,T)
        N, C, T = Xtr.shape
        X2 = np.transpose(Xtr, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
        scaler = StandardScaler()
        scaler.fit(X2)
        self.scaler = scaler
        return scaler

    def apply_scaler(self, scaler: StandardScaler = None):
        """Apply scaler to RAW windows and write into self.X as normalized windows (N,C,T)."""
        if scaler is None:
            scaler = getattr(self, "scaler", None)
        assert scaler is not None, "Scaler is not fitted. Call fit_scaler() first."

        X = self.X_raw
        N, C, T = X.shape
        X2 = np.transpose(X, (0, 2, 1)).reshape(-1, C)
        X2 = scaler.transform(X2)
        self.X = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)  # (N,C,T)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx: int):
        # returns (C,T) by default
        return (
            torch.from_numpy(self.X[idx]).float(),          # (C,T)
            torch.tensor(self.y[idx]).long(),               # ()
            int(self.subjects[idx]),
        )


class WISDMWindowDatasetForModel(Dataset):
    """
    Wraps WISDMDataset + indices, returns model input (T,C), y
    """
    def __init__(self, base: WISDMDataset, indices: np.ndarray):
        self.base = base
        self.indices = np.array(indices, dtype=np.int64)

    def __len__(self): return len(self.indices)

    def __getitem__(self, i: int):
        x_ct, y, _s = self.base[int(self.indices[i])]
        x_tc = x_ct.transpose(0, 1).contiguous()  # (T,C)
        return x_tc, y


def apply_scaler_to_array(X_raw_nct: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    """X_raw_nct: (N,C,T) -> normalized (N,C,T) using fitted scaler"""
    N, C, T = X_raw_nct.shape
    X2 = np.transpose(X_raw_nct, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
    X2 = scaler.transform(X2)
    Xn = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)
    return Xn


# ==============================================================================
# 5) Models (KEEP AS-IS)
# ==============================================================================
# ========================
# Model Components
# ========================
class BiLSTMBackbone(nn.Module):
    """BiLSTM backbone for all models
    Args:
        in_ch: input channels (WISDM: 3)
        d_model: output dimension (default: 128)
        hidden_dim: LSTM hidden dimension (default: 64, bidirectional -> 128)
        num_layers: number of LSTM layers (default: 2)
        dropout: dropout rate (default: 0.1)
    """
    def __init__(self, in_ch=27, d_model=128, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.lstm = nn.LSTM(
            input_size=in_ch,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.projection = nn.Linear(hidden_dim * 2, d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x is [B, T, C] already
        lstm_out, _ = self.lstm(x)                 # [B, T, 128]
        out = self.projection(lstm_out)            # [B, T, d_model]
        out = self.layer_norm(out)
        out = self.dropout(out)
        return out.transpose(1, 2)                 # [B, D, T]

# ========================
# GAP Model
# ========================
class GAPModel(nn.Module):
    """Baseline: Global Average Pooling"""
    def __init__(self, d_model=128, num_classes=12):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)                    # [B, D, T]
        features = fmap.transpose(1, 2)            # [B, T, D]
        pooled = features.mean(dim=1)              # [B, D]
        logits = self.fc(pooled)
        return logits

# ========================
# Pure-TPA
# ========================
class ProductionTPA(nn.Module):
    """Pure TPA"""
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1,
                 temperature=0.07):
        super().__init__()
        assert dim % heads == 0

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)

        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        P = self.num_prototypes

        x_norm = self.pre_norm(x)

        K = self.k_proj(x_norm)
        V = self.v_proj(x_norm)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, length):
            return t.view(B, length, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)
        Kh = split_heads(K, T)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)

        z_tpa = proto_tokens.mean(dim=1)
        z = self.fuse(z_tpa)
        return z

class TPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=12, tpa_config=None):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)                    # [B, D, T]
        features = fmap.transpose(1, 2)            # [B, T, D]
        z = self.tpa(features)                     # [B, D]
        logits = self.classifier(z)
        return logits

# ========================
# Gated-TPA
# ========================
class GatedTPAModel(nn.Module):
    def __init__(self, d_model=128, num_classes=12, tpa_config=None):
        super().__init__()
        self.backbone = BiLSTMBackbone(d_model=d_model)
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        fmap = self.backbone(x)                    # [B, D, T]
        features = fmap.transpose(1, 2)            # [B, T, D]

        z_gap = features.mean(dim=1)               # [B, D]
        z_tpa = self.tpa(features)                 # [B, D]

        gate_input = torch.cat([z_gap, z_tpa], dim=-1)
        g = self.gate(gate_input)

        z = g * z_gap + (1 - g) * z_tpa
        logits = self.classifier(z)
        return logits


# ---- Minimal wiring for WISDM (in_ch=3, num_classes=6) without changing model structure ----
def create_model(model_name: str, cfg: Config):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )

    if model_name == "GAP":
        model = GAPModel(d_model=cfg.d_model, num_classes=6).to(cfg.device)
        model.backbone = BiLSTMBackbone(in_ch=3, d_model=cfg.d_model).to(cfg.device)
        return model

    if model_name == "TPA":
        model = TPAModel(d_model=cfg.d_model, num_classes=6, tpa_config=tpa_config).to(cfg.device)
        model.backbone = BiLSTMBackbone(in_ch=3, d_model=cfg.d_model).to(cfg.device)
        return model

    if model_name == "Gated-TPA":
        model = GatedTPAModel(d_model=cfg.d_model, num_classes=6, tpa_config=tpa_config).to(cfg.device)
        model.backbone = BiLSTMBackbone(in_ch=3, d_model=cfg.d_model).to(cfg.device)
        return model

    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space)
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model (based on its clean F1)
# ==============================================================================
def run_wisdm_experiment(cfg: Config):
    # ---- Load RAW windows ----
    full_dataset = WISDMDataset(cfg.wisdm_txt_path, window_size=cfg.window_size, step_size=cfg.step_size)
    assert full_dataset.n_classes == 6, f"WISDM expected 6 classes, got {full_dataset.n_classes}"

    n_total = len(full_dataset)
    all_idx = np.arange(n_total)

    # ---- window-level split: train vs test (stratified) ----
    tr_idx, te_idx = train_test_split(
        all_idx,
        test_size=cfg.test_ratio,
        random_state=SEED,
        stratify=full_dataset.y
    )

    # ---- split train into train/val (stratified) ----
    tr2_idx, va_idx = train_test_split(
        tr_idx,
        test_size=cfg.val_ratio_within_train,
        random_state=SEED,
        stratify=full_dataset.y[tr_idx]
    )

    # ---- train-only scaler on RAW TRAIN windows ----
    scaler = full_dataset.fit_scaler(tr2_idx)
    full_dataset.apply_scaler(scaler)  # now full_dataset.X is normalized (N,C,T)

    # ---- loaders (model input: (T,C)) ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_ds = WISDMWindowDatasetForModel(full_dataset, tr2_idx)
    val_ds   = WISDMWindowDatasetForModel(full_dataset, va_idx)
    test_ds  = WISDMWindowDatasetForModel(full_dataset, te_idx)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (your exact list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # ---- Raw test (N,C,T) ----
    X_test_raw_nct = full_dataset.X_raw[te_idx].astype(np.float32)
    y_test = full_dataset.y[te_idx].astype(np.int64)
    rng = np.random.default_rng(SEED)

    def eval_model_on_perturbed(model, Xpert_raw_nct: np.ndarray) -> Tuple[float, float]:
        # RAW perturbed -> normalize with TRAIN scaler -> (N,C,T)
        Xnorm_nct = apply_scaler_to_array(Xpert_raw_nct, scaler)
        # -> (N,T,C) for model
        Xnorm_ntc = np.transpose(Xnorm_nct, (0, 2, 1)).astype(np.float32)

        ds = torch.utils.data.TensorDataset(
            torch.from_numpy(Xnorm_ntc).float(),
            torch.from_numpy(y_test).long()
        )
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("WISDM | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: BiLSTM")
    print(f"Splits: total={n_total} | train={len(tr2_idx)} | val={len(va_idx)} | test={len(te_idx)}")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_raw_nct, sp)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_raw_nct, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_wisdm_experiment(cfg)

Loaded WISDM dataset (single txt)
  X_raw shape   : (27108, 3, 80)  (N, C, T)
  y shape       : (27108,)  (N,)
  subjects shape: (27108,) (N,)
  num classes   : 6
  unique subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ... (total 36)

WISDM | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: BiLSTM
Splits: total=27108 | train=17348 | val=4338 | test=5422

--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.9602 | F1=0.9465 | (Val best F1=0.9359)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.8469 | F1=0.8427 | ΔF1=+0.1038
speed=0.90 | Acc=0.8383 | F1=0.8338 | ΔF1=+0.1127
speed=0.95 | Acc=0.8281 | F1=0.8262 | ΔF1=+0.1203
speed=1.05 | Acc=0.8082 | F1=0.8113 | ΔF1=+0.1352
speed=1.10 | Acc=0.7990 | F1=0.8047 | ΔF1=+0.1418
speed=1.15 | Acc=0.7909 | F1=0.7972 | ΔF1=+0.1493

[2

# Transformer

In [1]:
# -*- coding: utf-8 -*-
# WISDM (GAP vs TPA vs Gated-TPA) + Robustness (test-only) for EACH model
# - Train/Val/Test: train-only normalization (fit on TRAIN subset only)
# - Best model selection per architecture: best VAL macro-F1
# - Robustness: apply perturbation on RAW TEST only -> normalize(train stats) -> eval
# - Logging ONLY (no saving)
#
# Transformer backbone version
# - Model code: uses EXACT blocks you pasted (only num_classes/in_channels wired in create_model)
#
# IMPORTANT:
#   - I kept the Transformer model blocks as-is.
#   - Only changed: dataset/loader/normalization/splits to WISDM + create_model in_ch=3, n_cls=6.

import os, random, copy
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler


# ==============================================================================
# 1) Reproducibility
# ==============================================================================
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


# ==============================================================================
# 2) Config
# ==============================================================================
@dataclass
class Config:
    # ---- WISDM ----
    wisdm_txt_path: str = "/content/drive/MyDrive/Colab Notebooks/HAR/har_orig_datasets/WISDM_ar_v1.1_raw.txt"  # <-- CHANGE THIS
    window_size: int = 80
    step_size: int = 40

    test_ratio: float = 0.2
    val_ratio_within_train: float = 0.2  # val split from train portion

    epochs: int = 100
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    patience: int = 20
    min_delta: float = 1e-4

    # model dims
    d_model: int = 128

    # Transformer hyperparameters (match your pasted code defaults)
    num_layers: int = 2
    n_heads: int = 4
    ff_dim: int = 256
    dropout: float = 0.1
    max_seq_len: int = 200  # WISDM T=80 by default, so 200 is enough

    # TPA hyperparameters
    tpa_num_prototypes: int = 16
    tpa_heads: int = 4
    tpa_dropout: float = 0.1
    tpa_temperature: float = 0.07

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2

cfg = Config()


# ==============================================================================
# 3) WISDM Dataset (single txt) + train-only scaler
# ==============================================================================
class WISDMDataset(Dataset):
    """
    Creates windows from WISDM raw txt.

    Stores:
      - self.X_raw: (N, C, T) raw float32
      - self.X    : (N, C, T) working array (raw initially, becomes normalized after apply_scaler)
      - self.y    : (N,)
      - self.subjects : (N,)
    """
    def __init__(self, file_path: str, window_size: int = 80, step_size: int = 40):
        super().__init__()
        self.file_path = file_path
        self.window_size = window_size
        self.step_size = step_size

        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"WISDM txt file not found: {file_path}")

        df = self._load_file(file_path)
        X, y, s = self._create_windows(df)

        self.X_raw = X.astype(np.float32)             # (N,C,T)
        self.X = self.X_raw.copy().astype(np.float32) # will be normalized
        self.y = y.astype(np.int64)
        self.subjects = s.astype(np.int64)
        self.unique_subjects = sorted(np.unique(self.subjects))
        self.n_classes = int(len(np.unique(self.y)))

        print("=" * 80)
        print("Loaded WISDM dataset (single txt)")
        print(f"  X_raw shape   : {self.X_raw.shape}  (N, C, T)")
        print(f"  y shape       : {self.y.shape}  (N,)")
        print(f"  subjects shape: {self.subjects.shape} (N,)")
        print(f"  num classes   : {self.n_classes}")
        print(f"  unique subjects: {self.unique_subjects[:10]} ... (total {len(self.unique_subjects)})")
        print("=" * 80)

    def _load_file(self, file_path: str) -> pd.DataFrame:
        WISDM_LABEL_MAP = {
            "walking": 0,
            "jogging": 1,
            "sitting": 2,
            "standing": 3,
            "upstairs": 4,
            "downstairs": 5,
        }

        with open(file_path, "r") as f:
            lines = f.readlines()

        rows = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            line = line.replace(";", "")
            parts = line.split(",")

            if len(parts) != 6:
                continue

            subj, act, ts, x, y, z = parts
            if x.strip() == "" or y.strip() == "" or z.strip() == "":
                continue

            act_norm = act.strip().lower()
            if act_norm not in WISDM_LABEL_MAP:
                continue

            rows.append([subj, act_norm, ts, x, y, z])

        if not rows:
            raise ValueError(f"No valid rows parsed from file: {file_path}")

        df = pd.DataFrame(rows, columns=["subject", "activity", "timestamp", "x", "y", "z"])
        df = df.replace(["", "NaN", "nan"], np.nan).dropna(subset=["subject", "x", "y", "z"])

        df["subject"] = pd.to_numeric(df["subject"], errors="coerce")
        df["x"] = pd.to_numeric(df["x"], errors="coerce")
        df["y"] = pd.to_numeric(df["y"], errors="coerce")
        df["z"] = pd.to_numeric(df["z"], errors="coerce")
        df = df.dropna(subset=["subject", "x", "y", "z"])
        if df.empty:
            raise ValueError("After cleaning, WISDM DataFrame is empty. Check file format.")

        df["subject"] = df["subject"].astype(int)
        df["activity_id"] = df["activity"].map(WISDM_LABEL_MAP).astype(int)
        return df

    def _create_windows(self, df: pd.DataFrame):
        X_list, y_list, s_list = [], [], []

        for subj_id in sorted(df["subject"].unique()):
            df_sub = df[df["subject"] == subj_id]
            data = df_sub[["x", "y", "z"]].to_numpy(dtype=np.float32)  # (L,3)
            labels = df_sub["activity_id"].to_numpy(dtype=np.int64)    # (L,)
            L = len(df_sub)

            start = 0
            while start + self.window_size <= L:
                end = start + self.window_size
                window_x = data[start:end]     # (T,3)
                window_y = labels[end - 1]     # label at end

                X_list.append(window_x.T)      # (3,T)
                y_list.append(window_y)
                s_list.append(subj_id)
                start += self.step_size

        if len(X_list) == 0:
            raise ValueError("[WISDMDataset] No windows created. Try smaller window_size or check data.")

        X = np.stack(X_list, axis=0).astype(np.float32)  # (N,3,T)
        y = np.array(y_list, dtype=np.int64)
        s = np.array(s_list, dtype=np.int64)
        return X, y, s

    def fit_scaler(self, indices: np.ndarray) -> StandardScaler:
        """Fit StandardScaler on RAW train windows only."""
        Xtr = self.X_raw[indices]  # (N,C,T)
        N, C, T = Xtr.shape
        X2 = np.transpose(Xtr, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
        scaler = StandardScaler()
        scaler.fit(X2)
        self.scaler = scaler
        return scaler

    def apply_scaler(self, scaler: StandardScaler = None):
        """Apply scaler to RAW windows and write into self.X as normalized windows (N,C,T)."""
        if scaler is None:
            scaler = getattr(self, "scaler", None)
        assert scaler is not None, "Scaler is not fitted. Call fit_scaler() first."

        X = self.X_raw
        N, C, T = X.shape
        X2 = np.transpose(X, (0, 2, 1)).reshape(-1, C)
        X2 = scaler.transform(X2)
        self.X = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)  # (N,C,T)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx: int):
        # returns (C,T)
        return (
            torch.from_numpy(self.X[idx]).float(),
            torch.tensor(self.y[idx]).long(),
            int(self.subjects[idx]),
        )

class WISDMWindowDatasetForModel(Dataset):
    """
    Wraps WISDMDataset + indices, returns model input (T,C), y
    """
    def __init__(self, base: WISDMDataset, indices: np.ndarray):
        self.base = base
        self.indices = np.array(indices, dtype=np.int64)

    def __len__(self): return len(self.indices)

    def __getitem__(self, i: int):
        x_ct, y, _s = self.base[int(self.indices[i])]
        x_tc = x_ct.transpose(0, 1).contiguous()  # (T,C)
        return x_tc, y

def apply_scaler_to_array(X_raw_nct: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    """X_raw_nct: (N,C,T) -> normalized (N,C,T) using fitted scaler"""
    N, C, T = X_raw_nct.shape
    X2 = np.transpose(X_raw_nct, (0, 2, 1)).reshape(-1, C)  # (N*T, C)
    X2 = scaler.transform(X2)
    Xn = X2.reshape(N, T, C).transpose(0, 2, 1).astype(np.float32)
    return Xn


# ==============================================================================
# 4) Train-only Normalization function placeholders (UNCHANGED; WISDM uses scaler)
# ==============================================================================
def compute_channel_mean_std(X_ntc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = X_ntc.reshape(-1, X_ntc.shape[-1])  # (N*T, C)
    mean_c = X.mean(axis=0).astype(np.float32)
    std_c = (X.std(axis=0) + 1e-8).astype(np.float32)
    return mean_c, std_c

def normalize_ntc(X_ntc: np.ndarray, mean_c: np.ndarray, std_c: np.ndarray) -> np.ndarray:
    return ((X_ntc - mean_c[None, None, :]) / std_c[None, None, :]).astype(np.float32)

class PreloadedDataset(Dataset):
    def __init__(self, X_ntc: np.ndarray, y: np.ndarray):
        super().__init__()
        self.X = torch.from_numpy(X_ntc).float()  # [N, T, C]
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# ==============================================================================
# 5) Models (EXACT blocks you pasted)
# ==============================================================================
# ========================
# Transformer Backbone Components
# ========================
class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding"""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]
        Returns:
            [B, T, D]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerBackbone(nn.Module):
    """
    Lightweight Transformer Encoder Backbone
    - 2 layers
    - d_model=128
    - n_heads=4
    - ff_dim=256
    - Dropout=0.1
    """
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 max_seq_len: int = 200):
        super().__init__()

        self.d_model = d_model

        # Input projection: [B, C, T] -> [B, T, D]
        self.input_projection = nn.Linear(in_channels, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len, dropout)

        # Transformer Encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-LN for better stability
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Output normalization
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, C, T] - input sensor data
        Returns:
            [B, T, D] - transformed sequence
        """
        # [B, C, T] -> [B, T, C]
        # x = x.transpose(1, 2)

        # Project to d_model: [B, T, C] -> [B, T, D]
        x = self.input_projection(x)

        # Add positional encoding: [B, T, D]
        x = self.pos_encoder(x)

        # Transformer encoding: [B, T, D]
        x = self.transformer_encoder(x)

        # Final normalization: [B, T, D]
        x = self.norm(x)

        return x

# ========================
# GAP Model with Transformer
# ========================
class GAPModel(nn.Module):
    """Baseline: Global Average Pooling with Transformer Backbone"""
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout,
            max_seq_len=cfg.max_seq_len
        )
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]
        pooled = features.mean(dim=1)  # [B, D]
        logits = self.fc(pooled)
        return logits

# ========================
# Pure-TPA with Transformer
# ========================
class ProductionTPA(nn.Module):
    """Pure TPA"""
    def __init__(self, dim, num_prototypes=16, heads=4, dropout=0.1,
                 temperature=0.07):
        super().__init__()
        assert dim % heads == 0

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.num_prototypes = num_prototypes
        self.temperature = temperature

        self.proto = nn.Parameter(torch.randn(num_prototypes, dim) * 0.02)

        self.pre_norm = nn.LayerNorm(dim)

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        P = self.num_prototypes

        x_norm = self.pre_norm(x)

        K = self.k_proj(x_norm)
        V = self.v_proj(x_norm)
        Qp = self.q_proj(self.proto).unsqueeze(0).expand(B, -1, -1)

        def split_heads(t, length):
            return t.view(B, length, self.heads, self.head_dim).transpose(1, 2)

        Qh = split_heads(Qp, P)
        Kh = split_heads(K, T)
        Vh = split_heads(V, T)

        Qh = F.normalize(Qh, dim=-1)
        Kh = F.normalize(Kh, dim=-1)

        scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / self.temperature
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        proto_tokens = torch.matmul(attn, Vh)
        proto_tokens = proto_tokens.transpose(1, 2).contiguous().view(B, P, D)

        z_tpa = proto_tokens.mean(dim=1)

        z = self.fuse(z_tpa)

        return z

class TPAModel(nn.Module):
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12,
                 tpa_config=None):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout,
            max_seq_len=cfg.max_seq_len
        )
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]
        z = self.tpa(features)  # [B, D]
        logits = self.classifier(z)
        return logits

# ========================
# Gated-TPA with Transformer
# ========================
class GatedTPAModel(nn.Module):
    """Gated-TPA: Hybrid of TPA and GAP"""
    def __init__(self,
                 in_channels: int = 27,
                 d_model: int = 128,
                 num_layers: int = 2,
                 n_heads: int = 4,
                 ff_dim: int = 256,
                 dropout: float = 0.1,
                 num_classes: int = 12,
                 tpa_config=None):
        super().__init__()
        self.backbone = TransformerBackbone(
            in_channels=in_channels,
            d_model=d_model,
            num_layers=num_layers,
            n_heads=n_heads,
            ff_dim=ff_dim,
            dropout=dropout,
            max_seq_len=cfg.max_seq_len
        )
        self.tpa = ProductionTPA(
            dim=d_model,
            num_prototypes=tpa_config['num_prototypes'],
            heads=tpa_config['heads'],
            dropout=tpa_config['dropout'],
            temperature=tpa_config['temperature']
        )

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        features = self.backbone(x)  # [B, T, D]

        z_tpa = self.tpa(features)      # [B, D]
        z_gap = features.mean(dim=1)    # [B, D]

        gate_input = torch.cat([z_tpa, z_gap], dim=-1)  # [B, 2D]
        alpha = self.gate(gate_input)                   # [B, D]

        z = alpha * z_tpa + (1 - alpha) * z_gap
        logits = self.classifier(z)
        return logits


def create_model(model_name: str, cfg: Config):
    tpa_config = dict(
        num_prototypes=cfg.tpa_num_prototypes,
        heads=cfg.tpa_heads,
        dropout=cfg.tpa_dropout,
        temperature=cfg.tpa_temperature,
    )

    # WISDM: in_channels=3, num_classes=6
    in_ch = 3
    n_cls = 6

    if model_name == "GAP":
        return GAPModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls
        ).to(cfg.device)

    if model_name == "TPA":
        return TPAModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls,
            tpa_config=tpa_config
        ).to(cfg.device)

    if model_name == "Gated-TPA":
        return GatedTPAModel(
            in_channels=in_ch,
            d_model=cfg.d_model,
            num_layers=cfg.num_layers,
            n_heads=cfg.n_heads,
            ff_dim=cfg.ff_dim,
            dropout=cfg.dropout,
            num_classes=n_cls,
            tpa_config=tpa_config
        ).to(cfg.device)

    raise ValueError(f"Unknown model: {model_name}")


# ==============================================================================
# 6) Train / Eval (best by VAL macro-F1)
# ==============================================================================
def train_one_epoch(model, loader, opt, cfg: Config):
    model.train()
    for x, y in loader:
        x = x.to(cfg.device).float()  # [B, T, C]
        y = y.to(cfg.device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
        if torch.isnan(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

@torch.no_grad()
def evaluate(model, loader, cfg: Config) -> Tuple[float, float]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(cfg.device).float()
        y = y.to(cfg.device)
        logits = model(x)
        ps.append(logits.argmax(dim=-1).cpu().numpy())
        ys.append(y.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    return acc, f1

def train_model(model, train_loader, val_loader, cfg: Config) -> Dict:
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1.0
    best_wts = None
    patience_counter = 0

    for _ep in range(1, cfg.epochs + 1):
        train_one_epoch(model, train_loader, opt, cfg)
        _, val_f1 = evaluate(model, val_loader, cfg)

        if val_f1 > best_val_f1 + cfg.min_delta:
            best_val_f1 = val_f1
            best_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return {"best_val_f1": float(best_val_f1), "best_state_dict": best_wts}


# ==============================================================================
# 7) Robustness Perturbations (RAW space)
# ==============================================================================
def tempo_shift_one(x_ct: np.ndarray, speed: float) -> np.ndarray:
    C, T = x_ct.shape
    Tp = max(4, int(round(T / speed)))

    t_old = np.linspace(0, 1, T, dtype=np.float32)
    t_new = np.linspace(0, 1, Tp, dtype=np.float32)

    x_tp = np.zeros((C, Tp), dtype=np.float32)
    for c in range(C):
        x_tp[c] = np.interp(t_new, t_old, x_ct[c]).astype(np.float32)

    t_restore = np.linspace(0, 1, T, dtype=np.float32)
    x_out = np.zeros((C, T), dtype=np.float32)
    for c in range(C):
        x_out[c] = np.interp(t_restore, t_new, x_tp[c]).astype(np.float32)

    return x_out

def apply_tempo_shift_raw(X_raw_nct: np.ndarray, speed: float) -> np.ndarray:
    N = X_raw_nct.shape[0]
    out = np.empty_like(X_raw_nct, dtype=np.float32)
    for i in range(N):
        out[i] = tempo_shift_one(X_raw_nct[i], speed)
    return out

def apply_sensor_noise_raw(X_raw_nct: np.ndarray, mode: str, level: float, rng: np.random.Generator) -> np.ndarray:
    Xn = X_raw_nct.astype(np.float32).copy()
    sigma = Xn.transpose(1, 0, 2).reshape(Xn.shape[1], -1).std(axis=1) + 1e-8  # (C,)

    if mode == "gauss":
        noise = rng.normal(0.0, 1.0, size=Xn.shape).astype(np.float32)
        Xn = Xn + noise * (level * sigma[None, :, None])
    elif mode == "bias":
        b = (level * sigma).astype(np.float32)
        Xn = Xn + b[None, :, None]
    elif mode == "scale":
        Xn = Xn * (1.0 + float(level))
    else:
        raise ValueError(f"Unknown sensor noise mode: {mode}")
    return Xn


# ==============================================================================
# 8) Experiment: Train 3 models, then Robustness for EACH model
# ==============================================================================
def run_wisdm_experiment(cfg: Config):
    # ---- Load RAW windows ----
    full_dataset = WISDMDataset(cfg.wisdm_txt_path, window_size=cfg.window_size, step_size=cfg.step_size)
    assert full_dataset.n_classes == 6, f"WISDM expected 6 classes, got {full_dataset.n_classes}"

    n_total = len(full_dataset)
    all_idx = np.arange(n_total)

    # ---- train/test split (stratified by window label) ----
    tr_idx, te_idx = train_test_split(
        all_idx,
        test_size=cfg.test_ratio,
        random_state=SEED,
        stratify=full_dataset.y
    )

    # ---- train/val split inside train (stratified) ----
    tr2_idx, va_idx = train_test_split(
        tr_idx,
        test_size=cfg.val_ratio_within_train,
        random_state=SEED,
        stratify=full_dataset.y[tr_idx]
    )

    # ---- Train-only scaler on RAW TRAIN windows ----
    scaler = full_dataset.fit_scaler(tr2_idx)
    full_dataset.apply_scaler(scaler)  # now full_dataset.X is normalized (N,C,T)

    # ---- Build loaders (model input should be [B,T,C]) ----
    pin = (cfg.device == "cuda")
    g = torch.Generator(device="cpu").manual_seed(SEED)

    train_loader = DataLoader(WISDMWindowDatasetForModel(full_dataset, tr2_idx),
                              batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, generator=g, pin_memory=pin)
    val_loader   = DataLoader(WISDMWindowDatasetForModel(full_dataset, va_idx),
                              batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(WISDMWindowDatasetForModel(full_dataset, te_idx),
                              batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    # ---- Robust settings (your exact list) ----
    tempo_speeds = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
    gauss_levels = [0.05, 0.10, 0.20]
    bias_levels  = [0.05, 0.10, 0.20]
    scale_levels = [0.05, -0.05, 0.10, -0.10, 0.20, -0.20]

    # raw test as (N,C,T)
    X_test_nct_raw = full_dataset.X_raw[te_idx].astype(np.float32)
    y_test = full_dataset.y[te_idx].astype(np.int64)
    rng = np.random.default_rng(SEED)

    def eval_model_on_perturbed(model, Xpert_nct_raw: np.ndarray) -> Tuple[float, float]:
        # RAW perturbed -> normalize with TRAIN scaler -> (N,C,T)
        Xnorm_nct = apply_scaler_to_array(Xpert_nct_raw, scaler)
        # -> (N,T,C) for model forward
        Xnorm_ntc = np.transpose(Xnorm_nct, (0, 2, 1)).astype(np.float32)

        ds = torch.utils.data.TensorDataset(
            torch.from_numpy(Xnorm_ntc).float(),
            torch.from_numpy(y_test).long()
        )
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=pin)
        return evaluate(model, loader, cfg)

    # ---- Train + Robust per model ----
    model_names = ["GAP", "TPA", "Gated-TPA"]

    print("\n" + "="*80)
    print("WISDM | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: Transformer Encoder")
    print(f"Splits: total={n_total} | train={len(tr2_idx)} | val={len(va_idx)} | test={len(te_idx)}")
    print("="*80)

    for mn in model_names:
        set_seed(SEED)
        model = create_model(mn, cfg)

        train_info = train_model(model, train_loader, val_loader, cfg)
        clean_acc, clean_f1 = evaluate(model, test_loader, cfg)

        print("\n" + "-"*80)
        print(f"[{mn}]  Clean Test: Acc={clean_acc:.4f} | F1={clean_f1:.4f} | (Val best F1={train_info['best_val_f1']:.4f})")
        print("-"*80)

        # 1) Tempo
        print("[1) Temporal Scaling]")
        for sp in tempo_speeds:
            Xp = apply_tempo_shift_raw(X_test_nct_raw, sp)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"speed={sp:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 2) Gauss
        print("\n[2) Additive Gaussian Noise]")
        for lv in gauss_levels:
            Xp = apply_sensor_noise_raw(X_test_nct_raw, mode="gauss", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"σ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 3) Bias
        print("\n[3) Additive Bias Drift]")
        for lv in bias_levels:
            Xp = apply_sensor_noise_raw(X_test_nct_raw, mode="bias", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            print(f"δ={lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

        # 4) Scale
        print("\n[4) Multiplicative Scale Drift]")
        for lv in scale_levels:
            Xp = apply_sensor_noise_raw(X_test_nct_raw, mode="scale", level=lv, rng=rng)
            acc, f1 = eval_model_on_perturbed(model, Xp)
            df1 = clean_f1 - f1
            sign = "+" if lv > 0 else ""
            print(f"s={sign}{lv:>4.2f} | Acc={acc:.4f} | F1={f1:.4f} | ΔF1={df1:+.4f}")

    print("\n" + "="*80)
    print("DONE")
    print("="*80)


# ==============================================================================
# 9) Run
# ==============================================================================
if __name__ == "__main__":
    run_wisdm_experiment(cfg)

Loaded WISDM dataset (single txt)
  X_raw shape   : (27108, 3, 80)  (N, C, T)
  y shape       : (27108,)  (N,)
  subjects shape: (27108,) (N,)
  num classes   : 6
  unique subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ... (total 36)

WISDM | CLEAN + ROBUSTNESS (PER MODEL)  |  Backbone: Transformer Encoder
Splits: total=27108 | train=17348 | val=4338 | test=5422


/tmp/ipython-input-485/1917219793.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[GAP]  Clean Test: Acc=0.9655 | F1=0.9535 | (Val best F1=0.9461)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9489 | F1=0.9343 | ΔF1=+0.0192
speed=0.90 | Acc=0.9447 | F1=0.9304 | ΔF1=+0.0231
speed=0.95 | Acc=0.9393 | F1=0.9242 | ΔF1=+0.0293
speed=1.05 | Acc=0.9308 | F1=0.9159 | ΔF1=+0.0376
speed=1.10 | Acc=0.9334 | F1=0.9201 | ΔF1=+0.0334
speed=1.15 | Acc=0.9236 | F1=0.9116 | ΔF1=+0.0419

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9642 | F1=0.9517 | ΔF1=+0.0018
σ=0.10 | Acc=0.9603 | F1=0.9465 | ΔF1=+0.0070
σ=0.20 | Acc=0.9334 | F1=0.9003 | ΔF1=+0.0532

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9611 | F1=0.9445 | ΔF1=+0.0090
δ=0.10 | Acc=0.9498 | F1=0.9259 | ΔF1=+0.0276
δ=0.20 | Acc=0.8938 | F1=0.8398 | ΔF1=+0.1137

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9581 | F1=0.9410 | ΔF1=+0.0125
s=-0.05 | Acc=0.9646 | F1=0.9512 | ΔF1

/tmp/ipython-input-485/1917219793.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[TPA]  Clean Test: Acc=0.9672 | F1=0.9547 | (Val best F1=0.9507)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9456 | F1=0.9317 | ΔF1=+0.0231
speed=0.90 | Acc=0.9397 | F1=0.9251 | ΔF1=+0.0297
speed=0.95 | Acc=0.9388 | F1=0.9250 | ΔF1=+0.0298
speed=1.05 | Acc=0.9271 | F1=0.9139 | ΔF1=+0.0408
speed=1.10 | Acc=0.9259 | F1=0.9133 | ΔF1=+0.0414
speed=1.15 | Acc=0.9170 | F1=0.9053 | ΔF1=+0.0494

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9668 | F1=0.9546 | ΔF1=+0.0001
σ=0.10 | Acc=0.9642 | F1=0.9519 | ΔF1=+0.0029
σ=0.20 | Acc=0.9332 | F1=0.8919 | ΔF1=+0.0628

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9648 | F1=0.9496 | ΔF1=+0.0051
δ=0.10 | Acc=0.9554 | F1=0.9347 | ΔF1=+0.0200
δ=0.20 | Acc=0.9192 | F1=0.8914 | ΔF1=+0.0633

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9644 | F1=0.9503 | ΔF1=+0.0044
s=-0.05 | Acc=0.9672 | F1=0.9543 | ΔF1

/tmp/ipython-input-485/1917219793.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



--------------------------------------------------------------------------------
[Gated-TPA]  Clean Test: Acc=0.9579 | F1=0.9411 | (Val best F1=0.9351)
--------------------------------------------------------------------------------
[1) Temporal Scaling]
speed=0.85 | Acc=0.9163 | F1=0.8998 | ΔF1=+0.0413
speed=0.90 | Acc=0.9107 | F1=0.8942 | ΔF1=+0.0469
speed=0.95 | Acc=0.9045 | F1=0.8878 | ΔF1=+0.0532
speed=1.05 | Acc=0.8903 | F1=0.8752 | ΔF1=+0.0659
speed=1.10 | Acc=0.8838 | F1=0.8691 | ΔF1=+0.0720
speed=1.15 | Acc=0.8735 | F1=0.8639 | ΔF1=+0.0771

[2) Additive Gaussian Noise]
σ=0.05 | Acc=0.9594 | F1=0.9434 | ΔF1=-0.0023
σ=0.10 | Acc=0.9592 | F1=0.9447 | ΔF1=-0.0036
σ=0.20 | Acc=0.9382 | F1=0.9117 | ΔF1=+0.0294

[3) Additive Bias Drift]
δ=0.05 | Acc=0.9543 | F1=0.9315 | ΔF1=+0.0096
δ=0.10 | Acc=0.9471 | F1=0.9195 | ΔF1=+0.0216
δ=0.20 | Acc=0.9133 | F1=0.8685 | ΔF1=+0.0726

[4) Multiplicative Scale Drift]
s=+0.05 | Acc=0.9544 | F1=0.9336 | ΔF1=+0.0075
s=-0.05 | Acc=0.9583 | F1=0.9412